In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Student exam results

Clean the data, then answer the questions below.

- `subject` and `semester` have inconsistent casing; `passed` needs to become boolean.
- Add a `grade` column using `.apply()`: score ≥ 90 → `'A'`, ≥ 80 → `'B'`, ≥ 70 → `'C'`, else `'F'`.
- What is the grade distribution?
- Which subject has the higher average score? Use `np.mean`.
- What fraction of students passed? What is the standard deviation of scores? Use `np.std`.

In [9]:
scores = pd.DataFrame({
    'student':  ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Henry','Ivy','Jack'],
    'subject':  ['MATH','math','Science','SCIENCE','Math','science','Math','Science','MATH','science'],
    'score':    [88, 74, 91, 65, 83, 55, 95, 70, 78, 82],
    'semester': ['Fall-2023','FALL-2023','Spring-2024','fall-2023','Spring-2024',
                 'SPRING-2024','Fall-2023','fall-2023','Spring-2024','FALL-2023'],
    'passed':   ['Yes','YES','yes','No','Yes','No','yes','YES','Yes','yes'],
})

# Your code here

scores['subject'] = scores['subject'].str.upper()
scores['semester'] = scores['semester'].str.upper()
scores['passed'] = scores['passed'].str.upper().map({'YES': True, 'NO': False})

scores['grade'] = scores['score'].apply(lambda x: 'A' if x>=90 else
                                        'B' if x>=80 else
                                        'C' if x>=70 else
                                        'F')
print(scores['grade'].value_counts())
print(scores.groupby('subject')['score'].mean().idxmax(),'has the highest avg score')
print(scores['passed'].mean(),'passed ')
print(np.std(scores['score']), 'is the standard deviation')


grade
B    3
C    3
A    2
F    2
Name: count, dtype: int64
MATH has the highest avg score
0.8 passed 
11.734138229968147 is the standard deviation


---

## Level 2 — Sales rep performance

Named aggregation reminder — use `pd.NamedAgg` to build a summary table with custom column names:

```python
df.groupby('col').agg(
    total_rev  = pd.NamedAgg(column='revenue', aggfunc='sum'),
    avg_units  = pd.NamedAgg(column='units',   aggfunc='mean'),
    num_orders = pd.NamedAgg(column='units',   aggfunc='count'),
)
```

Use the `sales` data below:

1. Summarize by rep using `pd.NamedAgg`: total revenue, average units per order, number of orders.
2. Which rep has the highest average revenue per order? Compute it from your summary table and use `np.argmax`.
3. Summarize by region using dict-style agg: total revenue and max single-order revenue.
4. Use `np.argsort` to rank regions by total revenue from lowest to highest.

In [17]:
sales = pd.DataFrame({
    'rep':     ['Alice','Bob','Carol','Alice','Bob','Carol','Alice','Bob','Carol','Alice','Bob'],
    'region':  ['North','South','East','North','West','East','West','South','North','East','West'],
    'units':   [12, 8, 15, 10, 9, 14, 11, 7, 16, 13, 5],
    'revenue': [1440, 960, 1800, 1200, 1080, 1680, 1320, 840, 1920, 1560, 600],
    'month':   ['Jan','Jan','Jan','Feb','Feb','Feb','Mar','Mar','Mar','Apr','Apr'],
})

# Your code here

s = sales.groupby('rep').agg(
    total_rev = pd.NamedAgg('revenue','sum'),
    avg_units = ('units','mean'),
    num_orders = ('units','count')
)

s['rev_per_order'] = s['total_rev']/s['num_orders']
print(s['rev_per_order'].idxmax(),'has the highest')

sr = sales.groupby('region').agg(
    total_rev = ('revenue','sum'),
    max_so_rev = pd.NamedAgg(column = 'revenue', aggfunc = (lambda x: x.max()))
)

print(sr)
print(sr.index[np.argsort(sr['total_rev'])])

Carol has the highest
        total_rev  max_so_rev
region                       
East         5040        1800
North        4560        1920
South        1800         960
West         3000        1320
Index(['South', 'West', 'North', 'East'], dtype='object', name='region')


---

## Level 3 — Hotel bookings

Two tables: bookings and customers. Merge them, then answer the questions — no steps.

1. Merge on `customer_id`. Add a `total_cost` column (`nights × rate`).
2. Which customer (by name) has spent the most?
3. What is the average total cost by room type? By membership tier?
4. What are the 25th and 75th percentile total costs? Use `np.percentile`.
5. Which room type accounts for the highest total revenue overall?

In [25]:
bookings = pd.DataFrame({
    'booking_id':  ['B01','B02','B03','B04','B05','B06','B07','B08','B09','B10'],
    'customer_id': ['C01','C03','C01','C02','C04','C02','C03','C01','C04','C02'],
    'room_type':   ['Standard','Deluxe','Suite','Standard','Deluxe','Suite','Standard','Deluxe','Suite','Standard'],
    'nights':      [2, 3, 1, 4, 2, 3, 1, 2, 4, 3],
    'rate':        [120, 200, 350, 120, 200, 350, 120, 200, 350, 120],
})

customers = pd.DataFrame({
    'customer_id':  ['C01','C02','C03','C04'],
    'name':         ['Alice Park','Bob Chen','Carol Lee','Dave Kim'],
    'tier':         ['Gold','Silver','Gold','Bronze'],
    'member_since': [2019, 2021, 2020, 2023],
})

# Your code here
m = pd.merge(
    bookings, 
    customers, 
    on = 'customer_id',
    how = 'inner'
)

m['total_cost'] = m['nights']* m['rate']
print(m.groupby('name')['total_cost'].sum().idxmax(),'has spend the most')
print(m.groupby('room_type')['total_cost'].mean())
print(m.groupby('tier')['total_cost'].mean())
print(np.percentile(m['total_cost'],[25, 75]))
print(m.groupby('room_type')['total_cost'].sum().idxmax(),'accounts for the highest total revenue')

Bob Chen has spend the most
room_type
Deluxe      466.666667
Standard    300.000000
Suite       933.333333
Name: total_cost, dtype: float64
tier
Bronze    900.0
Gold      342.0
Silver    630.0
Name: total_cost, dtype: float64
[352.5 570. ]
Suite accounts for the highest total revenue
